> Check the signals in some clusters

In [ ]:
import pathlib

import pandas as pd

import plotly
import plotly.express as px
import plotly.graph_objects as go

import trapallada.analysis

from IPython.display import display, Markdown

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

So that *plotly* pictures can be exported to HTML

In [ ]:
plotly.offline.init_notebook_mode()

The base directory

In [ ]:
base_dir = trapallada.analysis.get_base_directory()
base_dir

# Input Files

Widget to choose a *sub*directory with results

In [ ]:
# results_dir = base_dir / 'results'
search_dir = base_dir / 'results' / 'normalization' / 'normalized_rate'
assert search_dir.exists()
combo = trapallada.analysis.make_combo_widget(search_dir)
display(combo)

Particular *sub*directory with the trained model. It should be a `str` so that parameters can be injected with papermill.

In [ ]:
model_dir = search_dir / combo.value
# ===============
# model_subdir = 'results/normalization/normalized_rate/fft_vanilla_x_sd_00025'
# model_subdir = 'results/normalization/normalized_rate/vanilla_x_sd_015_z_40'
# ===============
model_subdir = 'results/normalization/clean_0.9/vanilla_x_sd_015'

Just in case

In [ ]:
model_dir = base_dir / model_subdir
display(model_dir)
assert model_dir.exists()

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [ ]:
ds, positions_df, model, results_df, parameters, analysis_df = trapallada.analysis.load_results_dir(model_dir, analysis_also=True)

In [ ]:
analysis_df.head()

For the sake of convenience

In [ ]:
# pos_columns = ['RA', 'DEC', 'CRA', 'CDEC']
tsne_columns = [f'tsne_{i}' for i in range(1, 3)]

The number of objects for every *type*

In [ ]:
analysis_df['main_type'].value_counts()

The clustering (either *in projected space* or *in z*) to be analyzed

In [ ]:
cluster = 'cluster in projected space'
# cluster = 'cluster in z'

A few `main_type`s are selected

In [ ]:
on_plot_hover_columns = ['index', 'source', cluster, 'main_type']
on_plot_hover_columns

In [ ]:
for c in ['cluster in projected space', 'cluster in z']:
    analysis_df[cluster] = pd.Categorical(analysis_df[cluster])
    # display(analysis_df[c].value_counts())

The directory with the full signals is inferred

In [ ]:
full_signals_dir = (pathlib.Path.home() / parameters['dataset']['files']['signals_file']).parent / 'full_signals'

# All the signals

Empty `main_type`s are set to `UNKNOWN`

In [ ]:
analysis_df['main_type'].fillna('UNKNOWN', inplace=True)

In order to have fixed axes

In [ ]:
# analysis_df[tsne_columns].min(), analysis_df[tsne_columns].max()
xrange, yrange = list(zip(analysis_df[tsne_columns].min(), analysis_df[tsne_columns].max()))
xrange, yrange

One can **double-click** a `main_type` in the *Legend* to only show the corresponding curves

In [ ]:
analysis_df['main_type'].unique().shape

In [ ]:
analysis_df['signal']

In [ ]:
# trapallada.analysis.plot_window_within_signal(18786, full_signals_dir, analysis_df)

In [ ]:
# display(analysis_df.shape)
# analysis_df.loc[[31385, 31389], 'start'].unique()
# analysis_df.groupby(['start', 'end', 'signal']).size().sort_values(ascending=False)

All the windows are plotted, and some of them, considered *of interest*, are highlighted.

In [ ]:
tsne_plot = trapallada.analysis.TSNE(analysis_df, tsne_columns, full_signals_dir, on_plot_hover_columns)
fig = tsne_plot.make_fig()
tsne_plot.highlight_windows_of_interest()
fig.show()

# Zooming in

In [ ]:
regions = [
    {
        'x_range': [-10, 10],
        'y_range': [-10, 10],
    },
]

# regions = [
#     {
#         'x_range': (-10, 10),
#         'y_range': (-10, 10),
#     },
#     {
#         'x_range': (105, 125),
#         'y_range': (15, 30),
#     },
#     {
#         'x_range': (130, 134),
#         'y_range': (15, 20),
#     },
#     {
#         'x_range': (77, 85),
#         'y_range': (-88, -78),
#     },
#     {
#         'x_range': (50, 60),
#         'y_range': (-106, -94),
#     },
#     {
#         'x_range': (54, 62),
#         'y_range': (-118, -111),
#     },
#     {
#         'x_range': (88, 104),
#         'y_range': (-80, -66),
#     },
#     {
#         'x_range': (53, 59),
#         'y_range': (-88, -82),
#     },
#     {
#         'x_range': (-26, -10),
#         'y_range': (-120, -100),
#     },
#     {
#         'x_range': (-52, -42),
#         'y_range': (-94, -80),
#     },
# ]

# regions = [
#     {
#         'x_range': (-65, -40),
#         'y_range': (-35, -15),
#     },
#     {
#         'x_range': (6, 22),
#         'y_range': (-75, -50),
#     },
#     {
#         'x_range': (-70, -54),
#         'y_range': (0, 16),
#     },
#     {
#         'x_range': (-26, -16),
#         'y_range': (-72, -60),
#     },
#     {
#         'x_range': (-12, 6),
#         'y_range': (-48, -38),
#     },
# ]

regions += [
    {
        'x_range': [-65, -55],
        'y_range': [20, 35],
    },
    {
        'x_range': [30, 55],
        'y_range': [-38, -24],
    },
]

regions_file = model_dir / 'regions.yaml'

if regions_file.exists():
    regions = trapallada.analysis.load_regions(regions_file)
    print(f'Loaded {len(regions)} regions')


labels = [f'Region {i}' for i in range(len(regions))]

trapallada.analysis.make_save_button_widget(regions, regions_file)

In [ ]:
tsne_plot.inspect_regions(regions, labels, n_samples=5)
# tsne_plot.inspect_regions(regions, labels, dont_plot_signals=True)